---
# Labelling Emails
-----

## Set Up
-----

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandas as pd 

from nltk.corpus import stopwords
import spacy

import random

## Load Data
---

In [2]:
emails_df = pd.read_csv('../../data/cleaned_emails.csv', index_col= 0)

### Clean data

In [3]:
emails_df['all_text'] = emails_df['cleaned_subject'] + ' ' + emails_df['cleaned_body']

In [4]:
nlp = spacy.load('en_core_web_sm')

In [5]:
def remove_names(text):
    text = str(text)
    doc = nlp(text)
    for e in reversed(doc.ents):
        if e.label_ in ("PERSON", "ORG", "DATE",): 
            text = text[:e.start_char] + text[e.start_char + len(e.text):]

    return text

In [7]:
emails_df['all_text'] = emails_df['all_text'].apply(remove_names)

## Define Categories
---

- Finance
- Legal
- Operations
- Spam
- Personal

## Sampling 500 emails to label

In [ ]:
# get email tezt
emails = emails_df['all_text']

In [ ]:
# define keywords for each category
keywords = {
    'finance': {'budget', 'invoice', 'payment', 'expense', 'audit', 'billing', 'reimbursement',
                'cost', 'salary', 'finance', 'tax', 'receipt'},
    'legal': {'contract', 'agreement', 'compliance', 'lawsuit', 'legal', 'regulation',
              'policy', 'attorney', 'terms', 'dispute', 'settlement', 'confidentiality'},
    'operations': {'meeting', 'schedule', 'project', 'deadline', 'task', 'status', 'update',
                   'workflow', 'logistics', 'report', 'operation', 'planning'},
    'spam': {'free', 'winner', 'lottery', 'urgent', 'congratulations', 'prize',
             'discount', 'click', 'offer', 'limited time', 'buy now', 'subscribe'},
    'personal': {'party', 'dinner', 'family', 'birthday', 'holiday', 'weekend', 'travel',
                 'friend', 'congratulations', 'invite', 'love'}
}

In [ ]:
# function to add categories based on keywords
def find_categories(email_text):
    email_lower = email_text.lower()
    matched = []
    for category, words in keywords.items():
        if any(word in email_lower for word in words):
            matched.append(category)
    return matched if matched else ['unknown']


In [103]:
emails_categories = emails.apply(find_categories)
# in case of multiple categories, separate with a comma
emails_categories_str = emails_categories.apply(lambda x: ', '.join(x))

In [ ]:
full_df = pd.DataFrame({'email': emails, 'categories': emails_categories_str})

In [104]:
df = full_df.copy()

,email,categories
235873,POSREP 24 OCT HOEGH GALLEON 24Oct 1200LT at ...,unknown
78576,Refining LLC Yolanda In connection with the e...,"finance, legal"
182832,I will let you know any updates We will be al...,operations
140652,to Purchase Agreement LM6K2001 Kay This note...,"finance, legal, operations"
137577,Re DASH today Sorry but I dont know Probably i...,unknown
...,...,...
164676,RE vs Master Maybe If we truly believe this i...,legal
202929,smubetas RobyHMA is out of the office I will ...,unknown
128971,Capacity Michelle When you get a minute lets g...,legal
236781,RE EnEx New Online Gas Trading Platform andy t...,"legal, spam"


## Label Helper
----

Uuse this to chekc the labels and update if needed

In [26]:
samples_per_category = 250
sampled_indices = set()

for category in keywords.keys():
    category_indices = df[df['categories'].str.contains(category, case=False)].index.tolist()
    sampled = random.sample(category_indices, min(samples_per_category, len(category_indices)))
    sampled_indices.update(sampled)

# Create the manual review set
sampled_df = df.loc[list(sampled_indices)].copy()

# Save for manual labeling
sampled_df[['email', 'categories']].to_csv("label_review_set.csv", index=False)


In [85]:
import pandas as pd
from IPython.display import display, clear_output
import ipywidgets as widgets

df = pd.read_csv("label_review_set.csv")

labels = ['finance', 'legal', 'operations', 'spam', 'personal', 'unknown']
dropdown = widgets.Dropdown(options=labels, description='Label:')

i = 800
output = widgets.Output()

def show_email(index):
    clear_output(wait=True)
    
    email_html = f"""
    <div style='max-width: 800px; white-space: pre-wrap; font-family: monospace; padding: 10px; background-color: #f9f9f9; border: 1px solid #ccc;'>
        <strong>Email {index + 1} of {len(df)}</strong><br><br>
        {df.loc[index, 'email']}<br><br>
        <strong>Suggested categories:</strong> {df.loc[index, 'categories']}
    </div>
    """
    
    display(widgets.HTML(value=email_html))
    display(dropdown, next_button, save_button)

def on_next_clicked(b):
    global i
    df.at[i, 'final_label'] = dropdown.value
    i += 1
    if i < len(df):
        show_email(i)
    else:
        print("All emails reviewed! Exporting...")
        df.to_csv("labeled_emails.csv", index=False)

def on_save_clicked(b):
    df.to_csv("labeled_emails_progress.csv", index=False)
    print("Progress saved.")

next_button = widgets.Button(description="Next")
next_button.on_click(on_next_clicked)

save_button = widgets.Button(description="Save Progress")
save_button.on_click(on_save_clicked)

show_email(i)


HTML(value="\n    <div style='max-width: 800px; white-space: pre-wrap; font-family: monospace; padding: 10px; …

Dropdown(description='Label:', index=1, options=('finance', 'legal', 'operations', 'spam', 'personal', 'unknow…

Button(description='Next', style=ButtonStyle())

Button(description='Save Progress', style=ButtonStyle())

Progress saved.


In [100]:
df['final_label'].value_counts()

final_label
legal         203
operations    179
finance       139
personal      120
unknown       112
spam           64
Name: count, dtype: int64